# Freight Rate Prediction - ML Assessment

Gradient Boosting approach (LightGBM) with feature engineering and time-based validation.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE = 'D:/Assignments/Spotter.AI'
train_df = pd.read_csv(f'{BASE}/Data/train-test.csv')
validation_df = pd.read_csv(f'{BASE}/Data/validation.csv')
december_df = pd.read_csv(f'{BASE}/Data/december-chart-inputs.csv')
template_df = pd.read_csv(f'{BASE}/Data/validation-predictions-template.csv')

print(f"Training: {train_df.shape[0]:,} rows, {train_df.shape[1]} cols")
print(f"Validation: {validation_df.shape[0]:,} rows")
print(f"December: {december_df.shape[0]} rows")
print(f"\nTrain cols: {train_df.columns.tolist()}")
print(f"Valid cols: {validation_df.columns.tolist()}")

Training: 48,000 rows, 14 cols
Validation: 12,000 rows
December: 31 rows

Train cols: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']
Valid cols: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']


## EDA - Target Distribution

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(train_df['posted_rate'], bins=100, kde=True)
plt.title('Posted Rate Distribution')
plt.xlabel('Rate ($)')
plt.subplot(1, 2, 2)
sns.boxplot(y=train_df['posted_rate'])
plt.title('Posted Rate Box Plot')
plt.ylabel('Rate ($)')
plt.tight_layout()
plt.savefig(f'{BASE}/eda_posted_rate.png', dpi=150)
plt.show()
print(f"Mean: ${train_df['posted_rate'].mean():.2f}, Median: ${train_df['posted_rate'].median():.2f}")
print(f"Std: ${train_df['posted_rate'].std():.2f}, Range: ${train_df['posted_rate'].min():.2f} - ${train_df['posted_rate'].max():.2f}")

Mean: $2373.98, Median: $2030.76
Std: $1486.49, Range: $57.22 - $25533.00


## EDA - Equipment Analysis

In [3]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
train_df['equipment'].value_counts().plot(kind='bar', color=['#2196F3', '#4CAF50', '#FF9800'])
plt.title('Equipment Distribution')
plt.xlabel('Equipment')
plt.ylabel('Count')
plt.subplot(1, 2, 2)
sns.boxplot(data=train_df, x='equipment', y='posted_rate')
plt.title('Rate by Equipment')
plt.xlabel('Equipment')
plt.ylabel('Rate ($)')
plt.tight_layout()
plt.savefig(f'{BASE}/eda_equipment.png', dpi=150)
plt.show()
print(train_df.groupby('equipment')['posted_rate'].mean().sort_values(ascending=False))

equipment
Reefer     2553.636939
Flatbed    2445.087223
Dry Van    2271.548686
Name: posted_rate, dtype: float64


## EDA - Distance & Weight vs Rate

In [4]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(data=train_df, x='distance', y='posted_rate', alpha=0.1, s=10)
plt.title('Distance vs Rate')
plt.xlabel('Distance (miles)')
plt.ylabel('Rate ($)')
plt.subplot(1, 2, 2)
sns.scatterplot(data=train_df[train_df['weight'] > 0], x='weight', y='posted_rate', alpha=0.1, s=10)
plt.title('Weight vs Rate')
plt.xlabel('Weight (lbs)')
plt.ylabel('Rate ($)')
plt.tight_layout()
plt.savefig(f'{BASE}/eda_distance_weight.png', dpi=150)
plt.show()

## Data Cleaning

In [5]:
print(f"Negative weights in train: {(train_df['weight'] < 0).sum()}")
print(f"Missing weight in train: {train_df['weight'].isnull().sum()}")
print(f"Missing market_index in train: {train_df['market_index'].isnull().sum()}")
print(f"Missing distance in valid: {validation_df['distance'].isnull().sum()}")
print(f"Missing weight in valid: {validation_df['weight'].isnull().sum()}")

# Fix negative weights
train_df['weight'] = train_df['weight'].abs()
validation_df['weight'] = validation_df['weight'].abs()

# Impute missing weight by equipment median
weight_median = train_df.groupby('equipment')['weight'].median()
train_df['weight'] = train_df.apply(
    lambda r: r['weight'] if not pd.isna(r['weight']) else weight_median[r['equipment']], axis=1
)
validation_df['weight'] = validation_df.apply(
    lambda r: r['weight'] if not pd.isna(r['weight']) else weight_median[r['equipment']], axis=1
)

# Impute missing market_index
market_median = train_df['market_index'].median()
train_df['market_index'] = train_df['market_index'].fillna(market_median)
validation_df['market_index'] = validation_df['market_index'].fillna(market_median)

print(f"\nAfter cleaning - Missing weight train: {train_df['weight'].isnull().sum()}")
print(f"After cleaning - Missing market_index train: {train_df['market_index'].isnull().sum()}")

Negative weights in train: 292
Missing weight in train: 300
Missing market_index in train: 374
Missing distance in valid: 0
Missing weight in valid: 165

After cleaning - Missing weight train: 0
After cleaning - Missing market_index train: 0


## Feature Engineering

In [6]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def engineer_features(df):
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'])
    out['day_of_week'] = out['date'].dt.dayofweek
    out['month'] = out['date'].dt.month
    out['quarter'] = out['date'].dt.quarter
    out['day_of_month'] = out['date'].dt.day
    out['lat_diff'] = abs(out['pickup_lat'] - out['delivery_lat'])
    out['lon_diff'] = abs(out['pickup_lon'] - out['delivery_lon'])
    out['haversine_dist'] = haversine(out['pickup_lat'], out['pickup_lon'], out['delivery_lat'], out['delivery_lon'])
    out['weight_per_mile'] = out['weight'] / out['distance'].clip(lower=1)
    out['log_distance'] = np.log1p(out['distance'])
    out['log_weight'] = np.log1p(out['weight'])
    out['distance_x_weight'] = out['distance'] * out['weight']
    out['distance_x_market'] = out['distance'] * out['market_index']
    out['market_x_quote'] = out['market_index'] * out['quote_signal']
    out['distance_x_quote'] = out['distance'] * out['quote_signal']
    out['route'] = out['pickup'] + '_to_' + out['delivery']
    return out

train_df = engineer_features(train_df)
validation_df = engineer_features(validation_df)
december_df['date'] = pd.to_datetime(december_df['date'])
december_df['day_of_week'] = december_df['date'].dt.dayofweek
december_df['month'] = december_df['date'].dt.month
december_df['quarter'] = 4
december_df['day_of_month'] = december_df['date'].dt.day
december_df['lat_diff'] = 0.0
december_df['lon_diff'] = 0.0
december_df['haversine_dist'] = 360.0
december_df['weight_per_mile'] = december_df['weight'] / 360
december_df['log_distance'] = np.log1p(360)
december_df['log_weight'] = np.log1p(32000)
december_df['distance_x_weight'] = 360 * 32000
december_df['distance_x_market'] = 360 * december_df['market_index'] if 'market_index' in december_df.columns else 360 * market_median
december_df['market_x_quote'] = december_df['market_index'] * december_df['quote_signal'] if 'market_index' in december_df.columns and 'quote_signal' in december_df.columns else market_median * 2.0
december_df['distance_x_quote'] = 360 * december_df['quote_signal'] if 'quote_signal' in december_df.columns else 360 * 2.0
december_df['route'] = 'Lexington_to_Fort_Wayne'

print("Features engineered")

Features engineered


In [7]:
route_counts = train_df['route'].value_counts()
train_df['route_popularity'] = train_df['route'].map(route_counts)
validation_df['route_popularity'] = validation_df['route'].map(route_counts).fillna(0)
december_df['route_popularity'] = december_df['route'].map(route_counts).fillna(0)

city_origin_rate = train_df.groupby('pickup')['posted_rate'].mean()
city_dest_rate = train_df.groupby('delivery')['posted_rate'].mean()
train_df['origin_avg_rate'] = train_df['pickup'].map(city_origin_rate)
train_df['dest_avg_rate'] = train_df['delivery'].map(city_dest_rate)
validation_df['origin_avg_rate'] = validation_df['pickup'].map(city_origin_rate)
validation_df['dest_avg_rate'] = validation_df['delivery'].map(city_dest_rate)
december_df['origin_avg_rate'] = city_origin_rate.get('Lexington', train_df['posted_rate'].mean())
december_df['dest_avg_rate'] = city_dest_rate.get('Fort Wayne', train_df['posted_rate'].mean())

print(f"Route and city features added")
print(f"Unique routes: {train_df['route'].nunique()}")

Route and city features added
Unique routes: 4014


## Model Training & Comparison

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# Time-based split
split_date = pd.Timestamp('2025-09-01')
train_split = train_df[train_df['date'] < split_date].copy()
val_split = train_df[train_df['date'] >= split_date].copy()

print(f"Train split: {train_split.shape[0]:,} rows ({train_split['date'].min().date()} to {train_split['date'].max().date()})")
print(f"Val split: {val_split.shape[0]:,} rows ({val_split['date'].min().date()} to {val_split['date'].max().date()})")

Train split: 38,477 rows (2025-01-01 to 2025-08-31)
Val split: 9,523 rows (2025-09-01 to 2025-10-31)


In [9]:
drop_cols = ['load_id', 'pickup', 'delivery', 'route', 'date', 'posted_rate',
             'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon']
categorical_cols = ['equipment']
numerical_cols = [c for c in train_split.columns if c not in drop_cols + categorical_cols + ['posted_rate']]
print(f"Numerical features ({len(numerical_cols)}): {numerical_cols}")

train_encoded = pd.get_dummies(train_split, columns=categorical_cols, drop_first=True)
val_encoded = pd.get_dummies(val_split, columns=categorical_cols, drop_first=True)

for col in train_encoded.columns:
    if col not in val_encoded.columns and col != 'posted_rate':
        val_encoded[col] = 0
for col in val_encoded.columns:
    if col not in train_encoded.columns and col != 'posted_rate':
        train_encoded[col] = 0

feature_cols = [c for c in train_encoded.columns if c not in drop_cols + ['posted_rate', 'date']]
print(f"Total features: {len(feature_cols)}")

X_train = train_encoded[feature_cols]
y_train = train_encoded['posted_rate']
X_val = val_encoded[feature_cols]
y_val = val_encoded['posted_rate']

Numerical features (21): ['distance', 'weight', 'market_index', 'quote_signal', 'day_of_week', 'month', 'quarter', 'day_of_month', 'lat_diff', 'lon_diff', 'haversine_dist', 'weight_per_mile', 'log_distance', 'log_weight', 'distance_x_weight', 'distance_x_market', 'market_x_quote', 'distance_x_quote', 'route_popularity', 'origin_avg_rate', 'dest_avg_rate']
Total features: 23


In [10]:
# LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=1500, learning_rate=0.03, num_leaves=63, max_depth=7,
    min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1
)
lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
lgb_pred = lgb_model.predict(X_val)

print("=== LightGBM ===")
print(f"MAE: ${mean_absolute_error(y_val, lgb_pred):.2f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_val, lgb_pred)):.2f}")
print(f"R2: {r2_score(y_val, lgb_pred):.4f}")

=== LightGBM ===
MAE: $129.83
RMSE: $637.02
R2: 0.8258


In [12]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=1500, learning_rate=0.03, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbosity=0,
    early_stopping_rounds=50, eval_metric="rmse"
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_pred = xgb_model.predict(X_val)

print("=== XGBoost ===")
print(f"MAE: ${mean_absolute_error(y_val, xgb_pred):.2f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_val, xgb_pred)):.2f}")
print(f"R2: {r2_score(y_val, xgb_pred):.4f}")

=== XGBoost ===
MAE: $141.51
RMSE: $642.85
R2: 0.8225


In [13]:
# CatBoost
cb_model = CatBoostRegressor(
    iterations=1500, learning_rate=0.03, depth=7,
    loss_function='RMSE', random_seed=42, verbose=0
)
cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)
cb_pred = cb_model.predict(X_val)

print("=== CatBoost ===")
print(f"MAE: ${mean_absolute_error(y_val, cb_pred):.2f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_val, cb_pred)):.2f}")
print(f"R2: {r2_score(y_val, cb_pred):.4f}")

=== CatBoost ===
MAE: $122.71
RMSE: $635.00
R2: 0.8269


In [14]:
comparison = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'MAE': [mean_absolute_error(y_val, lgb_pred), mean_absolute_error(y_val, xgb_pred), mean_absolute_error(y_val, cb_pred)],
    'RMSE': [np.sqrt(mean_squared_error(y_val, lgb_pred)), np.sqrt(mean_squared_error(y_val, xgb_pred)), np.sqrt(mean_squared_error(y_val, cb_pred))],
    'R2': [r2_score(y_val, lgb_pred), r2_score(y_val, xgb_pred), r2_score(y_val, cb_pred)]
})
print("\n=== Model Comparison ===")
print(comparison.to_string(index=False))
best_model_name = comparison.loc[comparison['MAE'].idxmin(), 'Model']
print(f"\nBest model: {best_model_name}")


=== Model Comparison ===
   Model        MAE       RMSE       R2
LightGBM 129.832073 637.021801 0.825751
 XGBoost 141.509897 642.854979 0.822545
CatBoost 122.707655 635.002548 0.826854

Best model: CatBoost


## Train Final Model & Generate Predictions

In [15]:
if best_model_name == 'LightGBM':
    final_model = lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.03, num_leaves=63, max_depth=7,
        min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1
    )
elif best_model_name == 'XGBoost':
    final_model = xgb.XGBRegressor(
        n_estimators=1500, learning_rate=0.03, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbosity=0
    )
else:
    final_model = CatBoostRegressor(
        iterations=1500, learning_rate=0.03, depth=7,
        loss_function='RMSE', random_seed=42, verbose=0
    )

full_train_encoded = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
for col in full_train_encoded.columns:
    if col not in feature_cols and col != 'posted_rate':
        full_train_encoded[col] = 0

X_full_train = full_train_encoded[feature_cols]
y_full_train = full_train_encoded['posted_rate']
final_model.fit(X_full_train, y_full_train)
print(f"Final {best_model_name} trained on {len(X_full_train):,} samples")

Final CatBoost trained on 48,000 samples


In [ ]:
# Validation predictions
val_encoded_final = pd.get_dummies(validation_df, columns=categorical_cols, drop_first=True)
for col in feature_cols:
    if col not in val_encoded_final.columns:
        val_encoded_final[col] = 0

X_val_final = val_encoded_final[feature_cols]
val_predictions = final_model.predict(X_val_final)
val_predictions = np.maximum(val_predictions, 1.0)

output_df = pd.DataFrame({
    'load_id': validation_df['load_id'],
    'predicted_rate': np.round(val_predictions, 2)
})

print(f"Predictions: {output_df.shape[0]} rows")
print(f"Range: ${output_df['predicted_rate'].min():.2f} - ${output_df['predicted_rate'].max():.2f}")
print(f"Mean: ${output_df['predicted_rate'].mean():.2f}")
print(output_df.head(10))

output_df.to_csv(f'{BASE}/validation_predictions.csv', index=False)
print(f"\nSaved {BASE}/validation_predictions.csv")

In [16]:
# December predictions
december_encoded = pd.get_dummies(december_df, columns=['equipment'], drop_first=True)
for col in feature_cols:
    if col not in december_encoded.columns:
        december_encoded[col] = 0

X_december = december_encoded[feature_cols]
december_predictions = final_model.predict(X_december)
december_predictions = np.maximum(december_predictions, 1.0)

december_output = december_df.copy()
december_output['predicted_rate'] = np.round(december_predictions, 2)

print("December predictions:")
print(december_output.to_string())
print(f"\nRange: ${december_output['predicted_rate'].min():.2f} - ${december_output['predicted_rate'].max():.2f}")

december_output.to_csv(f'{BASE}/Data/december_chart_inputs.csv', index=False)
print(f"\nSaved {BASE}/Data/december_chart_inputs.csv")

December predictions:
       pickup    delivery  distance equipment  weight       date  predicted_rate  day_of_week  month  quarter  day_of_month  lat_diff  lon_diff  haversine_dist  weight_per_mile  log_distance  log_weight  distance_x_weight  distance_x_market  market_x_quote  distance_x_quote                    route  route_popularity  origin_avg_rate  dest_avg_rate
0   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-01          781.13            0     12        4             1       0.0       0.0           360.0        88.888889      5.888878   10.373522           11520000            380.088          2.1116             720.0  Lexington_to_Fort_Wayne               0.0      1824.140273    1981.883495
1   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-02          776.91            1     12        4             2       0.0       0.0           360.0        88.888889      5.888878   10.373522           11520000            380.088          2.1116             720.0  Le

In [17]:
# December chart preview
plt.figure(figsize=(12, 5))
plt.plot(december_output['date'], december_output['predicted_rate'], marker='o', linewidth=2, markersize=6)
plt.fill_between(december_output['date'], december_output['predicted_rate'], 
                 december_output['predicted_rate'].min() - 50, alpha=0.1)
plt.title('December 2025 Predicted Load Rate (Lexington -> Fort Wayne)')
plt.xlabel('Date')
plt.ylabel('Predicted Rate ($)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/december_predictions_preview.png', dpi=150)
plt.show()
print("Saved December preview chart")

Saved December preview chart


## Feature Importance

In [18]:
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
elif hasattr(final_model, 'get_feature_importance'):
    importances = final_model.get_feature_importance()

feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feat_imp.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(f'{BASE}/feature_importance.png', dpi=150)
plt.show()
print(feat_imp.head(10).to_string(index=False))

          feature  importance
     log_distance   17.028651
         distance   13.893142
   haversine_dist   11.969668
 distance_x_quote    9.972776
distance_x_market    7.045927
         lon_diff    5.043041
distance_x_weight    4.297626
 equipment_Reefer    3.687399
    dest_avg_rate    2.662300
     quote_signal    2.617769


## Summary

- **Best Model:** {best_model_name}
- **Validation MAE:** ${comparison.loc[comparison['Model'] == best_model_name, 'MAE'].values[0]:.2f}
- **Training:** 48,000 loads (Jan-Oct 2025)
- **Predictions:** 12,000 validation + 31 December days
- **Key features:** distance, weight, market_index, quote_signal, equipment, route
- **Cleaning:** Fixed negative weights, imputed missing values